In [1]:
import pandas as pd
import numpy as np
import random
import copy
import math
import warnings

warnings.filterwarnings('ignore')

In [2]:
# reno
# cr_df=pd.read_excel('project_CR_RENO (2).xlsx',sheet_name='WkDAY-Overall')
# df=pd.read_csv("elvisreno_od2023_weekday_export_odbc(new_001).csv")
# ke_df=pd.read_excel("RENO_KINGElvis (5).xlsx",sheet_name='Elvis_Review')

In [2]:
# cota files
cr_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkDAY-Overall')
df=pd.read_csv("elviscota2023obweekday_export_odbc(new_001).csv")
ke_df=pd.read_excel("COTA_KINGElvis (4).xlsx",sheet_name='Elvis_Review')

In [ ]:
wkday_overall_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkDAY-Overall')
# wkday_overall_df['LS_NAME_CODE']=wkday_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkday_route_df=pd.read_excel('VTA_CA_CR.xlsx',sheet_name='WkDAY-RouteTotal')

In [3]:
df.drop(columns=['ROUTE_SURVEYEDCode','ROUTE_SURVEYED'],inplace=True)
df.rename(columns={'ROUTE_SURVEYEDCode_New':'ROUTE_SURVEYEDCode','ROUTE_SURVEYED_NEW':'ROUTE_SURVEYED'},inplace=True) 

In [4]:
cr_df=pd.read_excel('project_CR_Santa Clarita.xlsx',sheet_name='WkDAY-Overall')
df=pd.read_csv("elvissantaclarita2023obweekday_export_odbc(v1).csv")
ke_df=pd.read_excel("SantaClarita_KINGElvis.xlsx",sheet_name='Elvis_Review')

In [5]:
ke_df=ke_df[ke_df['INTERV_INIT']!='999']
ke_df=ke_df[ke_df['INTERV_INIT']!=999]

In [6]:
ke_df=ke_df[ke_df['1st Cleaner']!='Test/No 5 MIN']

In [7]:
sample_df=pd.read_excel('2023 COTA OD Sampling Plan 20231019.xlsx',sheet_name='Sheet1',header=3)
sample_df

,Route #,Route Name,DIR #,Direction,AM Peak (3:00am-8:59am),Midday (9:00am-2:59pm),PM Peak (3:00pm-6:59pm),Late Night (7:00pm-2:59am),Total,Total Ridership,2,3,4,5,Total.1,Total Surveys,Unnamed: 16
0,1.0,KENNY/LIVINGSTON,1 - Dir=0,EAST/SOUTH,435.375038,949.494756,713.939258,222.864075,2321.673127,4673.595537,32.653128,71.212107,53.545444,16.714806,174.125485,607.567420,NaN
1,1.0,KENNY/LIVINGSTON,1 - Dir=1,WEST/NORTH,577.013556,955.907502,652.042310,166.959042,2351.922410,NaN,43.276017,71.693063,48.903173,12.521928,176.394181,NaN,NaN
2,2.0,E MAIN/N HIGH,2 - Dir=0,SOUTH/EAST,455.862428,1196.978893,686.535152,243.919091,2583.295565,5153.509602,34.189682,89.773417,51.490136,18.293932,193.747167,669.956248,NaN
3,2.0,E MAIN/N HIGH,2 - Dir=1,WEST/NORTH,501.236097,1159.461269,720.330065,189.186606,2570.214037,NaN,37.592707,86.959595,54.024755,14.188995,192.766053,NaN,NaN
4,3.0,NORTHWEST/HARRISBURG,3 - Dir=0,SOUTH,114.869824,168.130326,121.548725,76.754184,481.303060,1006.250565,8.615237,12.609774,9.116154,5.756564,36.09773,130.812573,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,101.0,CMAX,101 - Dir=1,SOUTH,381.670326,621.280514,377.300351,116.046419,1496.297611,NaN,28.625274,46.596039,28.297526,8.703481,112.222321,NaN,NaN
74,102.0,N HIGH/POLARIS PKWY,102 - Dir=0,NORTH,74.142638,107.598750,96.364748,33.848369,311.954505,617.215944,5.560698,8.069906,7.227356,2.538628,23.396588,80.238073,NaN
75,102.0,N HIGH/POLARIS PKWY,102 - Dir=1,SOUTH,87.505456,105.254850,74.277588,38.223545,305.261439,NaN,6.562909,7.894114,5.570819,2.866766,22.894608,NaN,NaN
76,NaN,NaN,NaN,NaN,7679.863770,13967.270715,9454.869287,3147.794390,34249.798163,34249.798163,575.989783,1047.545304,709.115197,236.084579,2568.734862,4452.473761,NaN


In [8]:
sample_df.columns

Index([                   'Route #',                 'Route Name',
                            'DIR #',                  'Direction',
         ' AM Peak (3:00am-8:59am)',     'Midday (9:00am-2:59pm)',
          'PM Peak (3:00pm-6:59pm)', 'Late Night (7:00pm-2:59am)',
                            'Total',            'Total Ridership',
                                  2,                            3,
                                  4,                            5,
                          'Total.1',              'Total Surveys',
                      'Unnamed: 16'],
      dtype='object')

In [9]:
sample_df['Route #'] = sample_df['Route #'].replace([np.inf, -np.inf], np.nan)
sample_df = sample_df.dropna(subset=['Route #'])
# sample_df['LS_NAME'] = (sample_df['Route #'].astype(int).astype(str) +' '+ sample_df['Route Name'])
sample_df['LS_NAME'] = sample_df['Route #'].astype(str)
sample_df

,Route #,Route Name,DIR #,Direction,AM Peak (3:00am-8:59am),Midday (9:00am-2:59pm),PM Peak (3:00pm-6:59pm),Late Night (7:00pm-2:59am),Total,Total Ridership,2,3,4,5,Total.1,Total Surveys,Unnamed: 16,LS_NAME
0,1.0,KENNY/LIVINGSTON,1 - Dir=0,EAST/SOUTH,435.375038,949.494756,713.939258,222.864075,2321.673127,4673.595537,32.653128,71.212107,53.545444,16.714806,174.125485,607.567420,NaN,1.0
1,1.0,KENNY/LIVINGSTON,1 - Dir=1,WEST/NORTH,577.013556,955.907502,652.042310,166.959042,2351.922410,NaN,43.276017,71.693063,48.903173,12.521928,176.394181,NaN,NaN,1.0
2,2.0,E MAIN/N HIGH,2 - Dir=0,SOUTH/EAST,455.862428,1196.978893,686.535152,243.919091,2583.295565,5153.509602,34.189682,89.773417,51.490136,18.293932,193.747167,669.956248,NaN,2.0
3,2.0,E MAIN/N HIGH,2 - Dir=1,WEST/NORTH,501.236097,1159.461269,720.330065,189.186606,2570.214037,NaN,37.592707,86.959595,54.024755,14.188995,192.766053,NaN,NaN,2.0
4,3.0,NORTHWEST/HARRISBURG,3 - Dir=0,SOUTH,114.869824,168.130326,121.548725,76.754184,481.303060,1006.250565,8.615237,12.609774,9.116154,5.756564,36.09773,130.812573,NaN,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,75.0,ARLINGTON/1ST AVE,75 - Dir=1,SOUTH,4.139860,0.000000,0.000000,0.000000,4.139860,NaN,0.310490,0.000000,0.000000,0.000000,0.31049,NaN,NaN,75.0
72,101.0,CMAX,101 - Dir=0,NORTH,276.968636,647.790898,481.579681,168.262186,1574.601402,3070.899013,20.772648,48.584317,36.118476,12.619664,118.095105,399.216872,NaN,101.0
73,101.0,CMAX,101 - Dir=1,SOUTH,381.670326,621.280514,377.300351,116.046419,1496.297611,NaN,28.625274,46.596039,28.297526,8.703481,112.222321,NaN,NaN,101.0
74,102.0,N HIGH/POLARIS PKWY,102 - Dir=0,NORTH,74.142638,107.598750,96.364748,33.848369,311.954505,617.215944,5.560698,8.069906,7.227356,2.538628,23.396588,80.238073,NaN,102.0


In [10]:
df['LS_NAME'] = df['ROUTE_SURVEYED'].apply(lambda x: str(x).split(' ')[0].strip() if str(x).split(' ')[0].strip() != 'nan' else np.nan)

In [11]:
df[df['LS_NAME']=='ZOO']['LS_NAME']

Series([], Name: LS_NAME, dtype: object)

In [12]:
sample_df[[2,3,4,5]]

,2,3,4,5
0,32.653128,71.212107,53.545444,16.714806
1,43.276017,71.693063,48.903173,12.521928
2,34.189682,89.773417,51.490136,18.293932
3,37.592707,86.959595,54.024755,14.188995
4,8.615237,12.609774,9.116154,5.756564
...,...,...,...,...
71,0.310490,0.000000,0.000000,0.000000
72,20.772648,48.584317,36.118476,12.619664
73,28.625274,46.596039,28.297526,8.703481
74,5.560698,8.069906,7.227356,2.538628


In [13]:
# cr_df[[1,2,3,4,5]]

In [14]:
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_FIELD,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS,LS_NAME
0,4983,2023-09-28 15:15:03,75,en,2023-09-28 15:09:47,2023-10-05 23:11:40,70.113.126.123,https://santaclarita-ca.etc-research.com/,-25200,999,...,\n\n,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,4984,2023-09-28 15:17:16,75,en,2023-09-28 15:15:06,2023-09-28 15:17:16,70.113.126.123,https://santaclarita-ca.etc-research.com/index...,-25200,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,5001,2023-10-02 11:05:11,75,en,2023-10-02 11:00:47,2023-10-02 11:05:11,174.243.146.36,https://santaclarita-ca.etc-research.com/,-25200,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,5004,2023-10-03 07:27:48,75,en,2023-10-03 07:21:04,2023-10-03 07:27:48,174.243.151.254,https://santaclarita-ca.etc-research.com/,-25200,LMG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14
4,5005,2023-10-03 07:34:04,75,en,2023-10-03 07:27:51,2023-10-03 07:34:04,174.243.151.254,https://santaclarita-ca.etc-research.com/index...,-25200,LMG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14


In [15]:
df[(df['ROUTE_SURVEYEDCode']=='COT_1_102_00')]['ROUTE_SURVEYED']

Series([], Name: ROUTE_SURVEYED, dtype: object)

In [16]:
ke_df.head(5)

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO
3,2024-01-12,5004,HereAPI,Vaishnavi,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-01-12,5005,HereAPI,Vaishnavi,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-01-12,5006,HereAPI,Vaishnavi,Use,NaN,NaN,Elvis_Review,Elvis_Review,good survey//O is out of area and used metroli...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2024-01-12,5009,HereAPI,Vaishnavi,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-01-12,5010,HereAPI,Vaishnavi,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
df.shape

(774, 725)

In [18]:
ke_df.shape

(738, 44)

In [19]:
ke_df=ke_df[ke_df['Final_Usage'].str.lower()=='use']
ke_df.shape

(698, 44)

In [20]:
df=pd.merge(df,ke_df['id'],on='id',how='inner')

In [21]:
df.drop_duplicates(subset='id',inplace=True)

In [22]:
df.shape

(698, 725)

In [23]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [24]:
cr_df

,SORT,LS_NAME_CODE,LS_NAME,Agency_CR,Pre-Early AM,Early AM,AM Peak,Midday,PM Peak,Evening,Direction Total,LS_ROUTELEVEL
0,0,SAN_1_1_01,1 - Outbound - To Castaic,SAN (placeholder),NaN,NaN,1.896765,2.592843,1.476375,1.071286,7.037268,11.064733
1,1,SAN_1_1_00,1 - Inbound - To McBean,SAN,NaN,NaN,0.610762,1.257072,1.533729,0.625901,4.027464,NaN
2,2,SAN_1_2_01,2 - Outbound - To Val Verde,SAN,NaN,NaN,1.597572,1.280851,1.016962,0.622157,4.517541,8.202162
3,3,SAN_1_2_00,2 - Inbound - To McBean,SAN,NaN,NaN,0.696928,1.017533,1.418951,0.551209,3.684621,NaN
4,4,SAN_1_3_01,3 - Outbound - To Seco Canyon,SAN,NaN,NaN,0.553204,0.676035,0.894075,0.884067,3.007381,6.083720
5,5,SAN_1_3_00,3 - Inbound - To Six Flags Magic Mountain,SAN,NaN,NaN,1.079804,0.939968,0.597340,0.459227,3.076340,NaN
6,6,SAN_1_4_01,4 - Outbound - To Newhall,SAN,NaN,NaN,1.690795,3.530684,2.373890,1.207284,8.802653,16.507443
7,7,SAN_1_4_00,4 - Inbound - To Bouquet Canyon/Plum Canyon,SAN,NaN,NaN,0.969202,2.955031,2.385797,1.394760,7.704790,NaN
8,8,SAN_1_5_01,5 - Outbound - To Vasquez Canyon,SAN,NaN,NaN,4.437076,6.241646,4.088108,4.109591,18.876421,37.668152
9,9,SAN_1_5_00,5 - Inbound - To Stevenson Ranch,SAN,NaN,NaN,4.548079,6.970255,4.649874,2.623523,18.791732,NaN


In [25]:
# cr_df[[0,1,2,3,4,5]]

In [26]:
sample_df[[2,3,4,5]]

,2,3,4,5
0,32.653128,71.212107,53.545444,16.714806
1,43.276017,71.693063,48.903173,12.521928
2,34.189682,89.773417,51.490136,18.293932
3,37.592707,86.959595,54.024755,14.188995
4,8.615237,12.609774,9.116154,5.756564
...,...,...,...,...
71,0.310490,0.000000,0.000000,0.000000
72,20.772648,48.584317,36.118476,12.619664
73,28.625274,46.596039,28.297526,8.703481
74,5.560698,8.069906,7.227356,2.538628


In [27]:
am_column=[2] #2 is for AM header
midday_colum=[3] #3 is for MIDDAY header
pm_column=[4] #4 is for PM header
evening_column=[5] #5 is for EVENING header

In [28]:
route_survey_column_check=['routesurveyedcode']
route_survey_column=check_all_characters_present(df,route_survey_column_check)
route_survey_column

['ROUTE_SURVEYEDCode']

In [29]:
value='REN_1_14_01'
'_'.join(value.split('_')[:-1])

'REN_1_14'

In [30]:
df['ROUTE_SURVEYED_SPLITED']=df[route_survey_column[0]].apply(lambda x: '_'.join(x.split('_')[:-1]))

In [31]:
df[df['ROUTE_SURVEYED_SPLITED']=='REN_1_2'].shape

(0, 726)

In [32]:
df[df['ROUTE_SURVEYEDCode']=='REN_1_2_01'].shape

(0, 726)

In [33]:
df[df['ROUTE_SURVEYEDCode']=='REN_1_2_00'].shape[0]

0

In [34]:
# time_column_check=['todcode']
time_column_check=['timeoncode']
time_column=check_all_characters_present(df,time_column_check)
time_column

['TIME_ONCode']

In [35]:
# df[(df[time_column[0]].isin(am_values))]

In [36]:
am_values=['AM1','AM2','AM3','AM4']
midday_values=['MID1','MID2','MID3','MID4','MID5']
pm_values=['MID6','PM1','PM2','PM3']
evening_values=['PM4','PM5','PM6','PM7']

In [37]:
df[(df['ROUTE_SURVEYEDCode']=='REN_1_VRGN_00')][['ROUTE_SURVEYEDCode']]

,ROUTE_SURVEYEDCode


In [38]:
cr_df.dropna(subset=['LS_NAME_CODE'],inplace=True)

In [39]:
len(cr_df['LS_NAME_CODE'].unique())

36

In [40]:
len(df['ROUTE_SURVEYEDCode'].unique())

33

In [41]:
am_column_check=['ampeak']
pm_column_check=['pmpeak']
midday_column_check=['midday']
evening_column_check=['evening']

# cota
# am_column_check=['am']
# pm_column_check=['pm']
# midday_column_check=['midday']
# evening_column_check=['eve']

am_column=check_all_characters_present(cr_df,am_column_check)
pm_column=check_all_characters_present(cr_df,pm_column_check)
midday_colum=check_all_characters_present(cr_df,midday_column_check)
evening_column=check_all_characters_present(cr_df,evening_column_check)
am_column,pm_column,midday_colum,evening_column

(['AM Peak'], ['PM Peak'], ['Midday'], ['Evening'])

In [42]:
new_df=pd.DataFrame()
new_df['ROUTE_SURVEYEDCode']=cr_df['LS_NAME_CODE']
new_df['CR_AM_Peak']=cr_df[am_column[0]]
new_df['CR_Midday']=cr_df[midday_colum[0]]
new_df['CR_PM_Peak']=cr_df[pm_column[0]]
new_df['CR_Evening']=cr_df[evening_column[0]]
new_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening
0,SAN_1_1_01,1.896765,2.592843,1.476375,1.071286
1,SAN_1_1_00,0.610762,1.257072,1.533729,0.625901
2,SAN_1_2_01,1.597572,1.280851,1.016962,0.622157
3,SAN_1_2_00,0.696928,1.017533,1.418951,0.551209
4,SAN_1_3_01,0.553204,0.676035,0.894075,0.884067
5,SAN_1_3_00,1.079804,0.939968,0.597340,0.459227
6,SAN_1_4_01,1.690795,3.530684,2.373890,1.207284
7,SAN_1_4_00,0.969202,2.955031,2.385797,1.394760
8,SAN_1_5_01,4.437076,6.241646,4.088108,4.109591
9,SAN_1_5_00,4.548079,6.970255,4.649874,2.623523


In [43]:
new_df.fillna(0,inplace=True)
new_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening
0,SAN_1_1_01,1.896765,2.592843,1.476375,1.071286
1,SAN_1_1_00,0.610762,1.257072,1.533729,0.625901
2,SAN_1_2_01,1.597572,1.280851,1.016962,0.622157
3,SAN_1_2_00,0.696928,1.017533,1.418951,0.551209
4,SAN_1_3_01,0.553204,0.676035,0.894075,0.884067
5,SAN_1_3_00,1.079804,0.939968,0.597340,0.459227
6,SAN_1_4_01,1.690795,3.530684,2.373890,1.207284
7,SAN_1_4_00,0.969202,2.955031,2.385797,1.394760
8,SAN_1_5_01,4.437076,6.241646,4.088108,4.109591
9,SAN_1_5_00,4.548079,6.970255,4.649874,2.623523


In [44]:
# for index, row in new_df.iterrows():
#     am_value=df[(df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode'])& (df[time_column[0]].isin(am_values))].shape[0]
#     am_value_ids=df[(df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode'])& (df[time_column[0]].isin(am_values))]['id'].values
#     midday_value=df[(df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode'])& (df[time_column[0]].isin(midday_values))].shape[0]
#     midday_value_ids=df[(df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode'])& (df[time_column[0]].isin(midday_values))]['id'].values
#     pm_value=df[(df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode'])& (df[time_column[0]].isin(pm_values))].shape[0]
#     pm_value_ids=df[(df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode'])& (df[time_column[0]].isin(pm_values))]['id'].values
#     evening_value=df[(df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode'])& (df[time_column[0]].isin(evening_values))].shape[0]
#     evening_value_ids=df[(df['ROUTE_SURVEYEDCode']==row['ROUTE_SURVEYEDCode'])& (df[time_column[0]].isin(evening_values))]['id'].values
#     new_df.loc[index,'CR_Total']=row['CR_AM_Peak']+row['CR_Midday']+row['CR_PM_Peak']+row['CR_Evening']
#     new_df.loc[index,'DB_AM_Peak']=am_value
#     new_df.loc[index,'DB_Midday']=midday_value
#     new_df.loc[index,'DB_PM_Peak']=pm_value
#     new_df.loc[index,'DB_Evening']=evening_value
#     new_df.loc[index,'DB_Total']=evening_value+am_value+midday_value+pm_value

#     new_df.loc[index,'DB_AM_IDS']=','.join(map(str,am_value_ids))
#     new_df.loc[index,'DB_Midday_IDS']=','.join(map(str,midday_value_ids))
#     new_df.loc[index,'DB_PM_IDS']=','.join(map(str,pm_value_ids))
#     new_df.loc[index,'DB_Evening_IDS']=','.join(map(str,evening_value_ids))
    

In [45]:
oppo_dir_time_column_check=['oppodirtriptimecode']
oppo_dir_time_column=check_all_characters_present(df,oppo_dir_time_column_check)
oppo_dir_time_column

['OPPO_DIR_TRIP_TIMECode']

In [46]:
for index, row in new_df.iterrows():
    route_code = row['ROUTE_SURVEYEDCode']
    
    # Define a function to get the counts and IDs
    def get_counts_and_ids(time_values):
        subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
        count = subset_df.shape[0]
        ids = subset_df['id'].values
        return count, ids
    
    # Calculate counts and IDs for each time slot
    am_value, am_value_ids = get_counts_and_ids(am_values)
    midday_value, midday_value_ids = get_counts_and_ids(midday_values)
    pm_value, pm_value_ids = get_counts_and_ids(pm_values)
    evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    
    # Assign values to new_df
    new_df.loc[index, 'CR_Total'] = row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
    new_df.loc[index, 'DB_AM_Peak'] = am_value
    new_df.loc[index, 'DB_Midday'] = midday_value
    new_df.loc[index, 'DB_PM_Peak'] = pm_value
    new_df.loc[index, 'DB_Evening'] = evening_value
    new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value
    
    # Join the IDs as a comma-separated string
    new_df.loc[index, 'DB_AM_IDS'] = ','.join(map(str, am_value_ids))
    new_df.loc[index, 'DB_Midday_IDS'] = ','.join(map(str, midday_value_ids))
    new_df.loc[index, 'DB_PM_IDS'] = ','.join(map(str, pm_value_ids))
    new_df.loc[index, 'DB_Evening_IDS'] = ','.join(map(str, evening_value_ids))


In [47]:
new_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS
0,SAN_1_1_01,1.896765,2.592843,1.476375,1.071286,7.037268,0.0,8.0,8.0,5.0,21.0,,"5105,5157,5273,5275,5276,5277,5290,5291","5127,5292,5301,5302,5311,5312,5314,5795","5321,5323,5324,5325,5326"
1,SAN_1_1_00,0.610762,1.257072,1.533729,0.625901,4.027464,0.0,2.0,7.0,2.0,11.0,,"5278,5287","5307,5308,5309,5310,5317,5319,5320","5327,5328"
2,SAN_1_2_01,1.597572,1.280851,1.016962,0.622157,4.517541,0.0,9.0,6.0,1.0,16.0,,"5107,5148,5611,5679,5800,5835,5916,5935,5936","5158,6003,6004,6006,6008,6009",5967
3,SAN_1_2_00,0.696928,1.017533,1.418951,0.551209,3.684621,0.0,9.0,0.0,1.0,10.0,,"5143,5841,5842,5919,5921,5922,5923,5942,5943",,5966
4,SAN_1_3_01,0.553204,0.676035,0.894075,0.884067,3.007381,0.0,4.0,3.0,0.0,7.0,,"5819,5993,5994,5995","5848,5849,5850",
5,SAN_1_3_00,1.079804,0.939968,0.597340,0.459227,3.076340,0.0,9.0,2.0,0.0,11.0,,"5043,5052,5054,5151,5439,5441,5628,5827,5992","5128,5901",
6,SAN_1_4_01,1.690795,3.530684,2.373890,1.207284,8.802653,0.0,9.0,14.0,0.0,23.0,,"5109,5110,5626,5634,5970,5972,5973,5985,5988","5121,5125,5515,5517,5518,5651,5652,5653,5742,5...",
7,SAN_1_4_00,0.969202,2.955031,2.385797,1.394760,7.704790,0.0,21.0,12.0,0.0,33.0,,"5104,5111,5616,5617,5627,5629,5630,5632,5633,5...","5124,5512,5514,5646,5647,5952,5955,5956,5957,5...",
8,SAN_1_5_01,4.437076,6.241646,4.088108,4.109591,18.876421,4.0,30.0,14.0,4.0,52.0,"5058,5059,5060,5522","5073,5074,5100,5530,5532,5546,5548,5549,5550,5...","5123,5170,5478,5566,5567,5568,5569,5570,5572,5...","5592,5593,5594,5766"
9,SAN_1_5_00,4.548079,6.970255,4.649874,2.623523,18.791732,0.0,31.0,12.0,4.0,47.0,,"5062,5068,5069,5075,5076,5459,5460,5461,5463,5...","5240,5241,5243,5246,5247,5479,5480,5481,5573,5...","5588,5589,5590,5591"


In [48]:
# ('_').join("REN_1_LNCL_01".split('_')[:-1])

In [49]:
new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )

In [50]:
new_df[['ROUTE_SURVEYEDCode','ROUTE_SURVEYEDCode_Splited']]

,ROUTE_SURVEYEDCode,ROUTE_SURVEYEDCode_Splited
0,SAN_1_1_01,SAN_1_1
1,SAN_1_1_00,SAN_1_1
2,SAN_1_2_01,SAN_1_2
3,SAN_1_2_00,SAN_1_2
4,SAN_1_3_01,SAN_1_3
5,SAN_1_3_00,SAN_1_3
6,SAN_1_4_01,SAN_1_4
7,SAN_1_4_00,SAN_1_4
8,SAN_1_5_01,SAN_1_5
9,SAN_1_5_00,SAN_1_5


In [51]:
route_level_df=pd.DataFrame()
route_level_df

""


In [52]:
unique_routes=new_df['ROUTE_SURVEYEDCode_Splited'].unique()
unique_routes

array(['SAN_1_1', 'SAN_1_2', 'SAN_1_3', 'SAN_1_4', 'SAN_1_5', 'SAN_1_6',
       'SAN_1_7', 'SAN_1_12', 'SAN_1_14', 'SAN_1_757', 'SAN_1_501',
       'SAN_1_502', 'SAN_1_791', 'SAN_1_792', 'SAN_1_794', 'SAN_1_796',
       'SAN_1_797', 'SAN_1_799'], dtype=object)

In [53]:
route_level_df['ROUTE_SURVEYEDCode']=unique_routes
route_level_df

,ROUTE_SURVEYEDCode
0,SAN_1_1
1,SAN_1_2
2,SAN_1_3
3,SAN_1_4
4,SAN_1_5
5,SAN_1_6
6,SAN_1_7
7,SAN_1_12
8,SAN_1_14
9,SAN_1_757


In [54]:
for route in unique_routes:
    subset_df=new_df[new_df['ROUTE_SURVEYEDCode_Splited']==route]
    sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
    sum_per_route_db = subset_df[['DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()
    print(','.join(str(value) for value in subset_df['DB_AM_IDS'].values))
    break

,


In [55]:
for index , row in route_level_df.iterrows():
    subset_df=new_df[new_df['ROUTE_SURVEYEDCode_Splited']==row['ROUTE_SURVEYEDCode']]
    sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
    sum_per_route_db = subset_df[['DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()
    
    route_level_df.loc[index,'CR_AM_Peak']=sum_per_route_cr['CR_AM_Peak']
    route_level_df.loc[index,'CR_Midday']=sum_per_route_cr['CR_Midday']
    route_level_df.loc[index,'CR_PM_Peak']=sum_per_route_cr['CR_PM_Peak']
    route_level_df.loc[index,'CR_Evening']=sum_per_route_cr['CR_Evening']
    route_level_df.loc[index,'CR_Total']=sum_per_route_cr['CR_Total']
    
    route_level_df.loc[index,'DB_AM_Peak']=sum_per_route_db['DB_AM_Peak']
    route_level_df.loc[index,'DB_Midday']=sum_per_route_db['DB_Midday']
    route_level_df.loc[index,'DB_PM_Peak']=sum_per_route_db['DB_PM_Peak']
    route_level_df.loc[index,'DB_Evening']=sum_per_route_db['DB_Evening']
    route_level_df.loc[index,'DB_Total']=sum_per_route_db['DB_Total']   
    route_level_df.loc[index,'DB_AM_IDS']=', '.join(str(value) for value in subset_df['DB_AM_IDS'].values)    
    route_level_df.loc[index,'DB_Midday_IDS']=', '.join(str(value) for value in subset_df['DB_Midday_IDS'].values)    
    route_level_df.loc[index,'DB_PM_IDS']=', '.join(str(value) for value in subset_df['DB_PM_IDS'].values)    
    route_level_df.loc[index,'DB_Evening_IDS']=', '.join(str(value) for value in subset_df['DB_Evening_IDS'].values)

In [56]:
route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,0.0,10.0,15.0,7.0,32.0,",","5105,5157,5273,5275,5276,5277,5290,5291, 5278,...","5127,5292,5301,5302,5311,5312,5314,5795, 5307,...","5321,5323,5324,5325,5326, 5327,5328"
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,0.0,18.0,6.0,2.0,26.0,",","5107,5148,5611,5679,5800,5835,5916,5935,5936, ...","5158,6003,6004,6006,6008,6009,","5967, 5966"
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,0.0,13.0,5.0,0.0,18.0,",","5819,5993,5994,5995, 5043,5052,5054,5151,5439,...","5848,5849,5850, 5128,5901",","
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,0.0,30.0,26.0,0.0,56.0,",","5109,5110,5626,5634,5970,5972,5973,5985,5988, ...","5121,5125,5515,5517,5518,5651,5652,5653,5742,5...",","
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,4.0,61.0,26.0,8.0,99.0,"5058,5059,5060,5522,","5073,5074,5100,5530,5532,5546,5548,5549,5550,5...","5123,5170,5478,5566,5567,5568,5569,5570,5572,5...","5592,5593,5594,5766, 5588,5589,5590,5591"
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,2.0,63.0,48.0,13.0,126.0,", 5668,5670","5064,5065,5066,5067,5077,5106,5155,5210,5227,5...","5180,5249,5250,5482,5484,5637,5640,5641,5642,5...","5254,5255,5256,5257,5258,5485,5487,5488,5869, ..."
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,0.0,16.0,3.0,1.0,20.0,",","5044,5045,5050,5051,5053,5055,5056,5057,5631,5...","5903, 5129,5856",", 5126"
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,2.0,43.0,27.0,15.0,87.0,"5369,5601,","5150,5152,5383,5400,5403,5404,5406,5445,5496,5...","5120,5182,5419,5420,5426,5428,5430,5477,5584,5...","5597,5655,5658, 5433,5435,5436,5660,5661,5758,..."
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,0.0,23.0,21.0,3.0,47.0,",","5009,5010,5020,5021,5022,5341,5342,5344,5349,5...","5023,5024,5030,5031,5032,5033,5101,5356,5790,5...","5909, 5908,5910"
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,0.0,30.0,5.0,8.0,43.0,",","5085,5086,5089,5090,5135,5136,5563,5696,5697,5...","5579,5580, 5117,5159,5581",", 5134,5583,5656,5767,6023,6024,6025,6027"


In [57]:
for index, row in route_level_df.iterrows():
    route_code = row['ROUTE_SURVEYEDCode']
    subset_df = new_df[new_df['ROUTE_SURVEYEDCode_Splited'] == route_code]
    
    cr_columns = ['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening', 'CR_Total']
    db_columns = ['DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening', 'DB_Total']
    
    sum_per_route_cr = subset_df[cr_columns].sum()
    sum_per_route_db = subset_df[db_columns].sum()
    
    route_level_df.loc[index, cr_columns] = sum_per_route_cr.values
    route_level_df.loc[index, db_columns] = sum_per_route_db.values
route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,0.0,10.0,15.0,7.0,32.0,",","5105,5157,5273,5275,5276,5277,5290,5291, 5278,...","5127,5292,5301,5302,5311,5312,5314,5795, 5307,...","5321,5323,5324,5325,5326, 5327,5328"
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,0.0,18.0,6.0,2.0,26.0,",","5107,5148,5611,5679,5800,5835,5916,5935,5936, ...","5158,6003,6004,6006,6008,6009,","5967, 5966"
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,0.0,13.0,5.0,0.0,18.0,",","5819,5993,5994,5995, 5043,5052,5054,5151,5439,...","5848,5849,5850, 5128,5901",","
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,0.0,30.0,26.0,0.0,56.0,",","5109,5110,5626,5634,5970,5972,5973,5985,5988, ...","5121,5125,5515,5517,5518,5651,5652,5653,5742,5...",","
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,4.0,61.0,26.0,8.0,99.0,"5058,5059,5060,5522,","5073,5074,5100,5530,5532,5546,5548,5549,5550,5...","5123,5170,5478,5566,5567,5568,5569,5570,5572,5...","5592,5593,5594,5766, 5588,5589,5590,5591"
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,2.0,63.0,48.0,13.0,126.0,", 5668,5670","5064,5065,5066,5067,5077,5106,5155,5210,5227,5...","5180,5249,5250,5482,5484,5637,5640,5641,5642,5...","5254,5255,5256,5257,5258,5485,5487,5488,5869, ..."
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,0.0,16.0,3.0,1.0,20.0,",","5044,5045,5050,5051,5053,5055,5056,5057,5631,5...","5903, 5129,5856",", 5126"
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,2.0,43.0,27.0,15.0,87.0,"5369,5601,","5150,5152,5383,5400,5403,5404,5406,5445,5496,5...","5120,5182,5419,5420,5426,5428,5430,5477,5584,5...","5597,5655,5658, 5433,5435,5436,5660,5661,5758,..."
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,0.0,23.0,21.0,3.0,47.0,",","5009,5010,5020,5021,5022,5341,5342,5344,5349,5...","5023,5024,5030,5031,5032,5033,5101,5356,5790,5...","5909, 5908,5910"
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,0.0,30.0,5.0,8.0,43.0,",","5085,5086,5089,5090,5135,5136,5563,5696,5697,5...","5579,5580, 5117,5159,5581",", 5134,5583,5656,5767,6023,6024,6025,6027"


In [58]:
for index, row in route_level_df.iterrows():
    am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
    midday_diff=row['CR_Midday']-row['DB_Midday']    
    pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
    evening_diff=row['CR_Evening']-row['DB_Evening']
    total_diff=row['CR_Total']-row['DB_Total']
    
    # Check if the difference is negative, and set to 0 if true
    route_level_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
    route_level_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
    route_level_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
    route_level_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
    route_level_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))

In [59]:
route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,0.0,10.0,15.0,7.0,32.0,",","5105,5157,5273,5275,5276,5277,5290,5291, 5278,...","5127,5292,5301,5302,5311,5312,5314,5795, 5307,...","5321,5323,5324,5325,5326, 5327,5328",3.0,0.0,0.0,0.0,3.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,0.0,18.0,6.0,2.0,26.0,",","5107,5148,5611,5679,5800,5835,5916,5935,5936, ...","5158,6003,6004,6006,6008,6009,","5967, 5966",3.0,0.0,0.0,0.0,3.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,0.0,13.0,5.0,0.0,18.0,",","5819,5993,5994,5995, 5043,5052,5054,5151,5439,...","5848,5849,5850, 5128,5901",",",2.0,0.0,0.0,2.0,4.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,0.0,30.0,26.0,0.0,56.0,",","5109,5110,5626,5634,5970,5972,5973,5985,5988, ...","5121,5125,5515,5517,5518,5651,5652,5653,5742,5...",",",3.0,0.0,0.0,3.0,6.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,4.0,61.0,26.0,8.0,99.0,"5058,5059,5060,5522,","5073,5074,5100,5530,5532,5546,5548,5549,5550,5...","5123,5170,5478,5566,5567,5568,5569,5570,5572,5...","5592,5593,5594,5766, 5588,5589,5590,5591",5.0,0.0,0.0,0.0,5.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,2.0,63.0,48.0,13.0,126.0,", 5668,5670","5064,5065,5066,5067,5077,5106,5155,5210,5227,5...","5180,5249,5250,5482,5484,5637,5640,5641,5642,5...","5254,5255,5256,5257,5258,5485,5487,5488,5869, ...",11.0,0.0,0.0,0.0,11.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,0.0,16.0,3.0,1.0,20.0,",","5044,5045,5050,5051,5053,5055,5056,5057,5631,5...","5903, 5129,5856",", 5126",1.0,0.0,0.0,0.0,1.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,2.0,43.0,27.0,15.0,87.0,"5369,5601,","5150,5152,5383,5400,5403,5404,5406,5445,5496,5...","5120,5182,5419,5420,5426,5428,5430,5477,5584,5...","5597,5655,5658, 5433,5435,5436,5660,5661,5758,...",6.0,0.0,0.0,0.0,6.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,0.0,23.0,21.0,3.0,47.0,",","5009,5010,5020,5021,5022,5341,5342,5344,5349,5...","5023,5024,5030,5031,5032,5033,5101,5356,5790,5...","5909, 5908,5910",3.0,0.0,0.0,0.0,3.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,0.0,30.0,5.0,8.0,43.0,",","5085,5086,5089,5090,5135,5136,5563,5696,5697,5...","5579,5580, 5117,5159,5581",", 5134,5583,5656,5767,6023,6024,6025,6027",3.0,0.0,0.0,0.0,3.0


In [60]:
compare_df=route_level_df
compare_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,0.0,10.0,15.0,7.0,32.0,",","5105,5157,5273,5275,5276,5277,5290,5291, 5278,...","5127,5292,5301,5302,5311,5312,5314,5795, 5307,...","5321,5323,5324,5325,5326, 5327,5328",3.0,0.0,0.0,0.0,3.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,0.0,18.0,6.0,2.0,26.0,",","5107,5148,5611,5679,5800,5835,5916,5935,5936, ...","5158,6003,6004,6006,6008,6009,","5967, 5966",3.0,0.0,0.0,0.0,3.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,0.0,13.0,5.0,0.0,18.0,",","5819,5993,5994,5995, 5043,5052,5054,5151,5439,...","5848,5849,5850, 5128,5901",",",2.0,0.0,0.0,2.0,4.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,0.0,30.0,26.0,0.0,56.0,",","5109,5110,5626,5634,5970,5972,5973,5985,5988, ...","5121,5125,5515,5517,5518,5651,5652,5653,5742,5...",",",3.0,0.0,0.0,3.0,6.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,4.0,61.0,26.0,8.0,99.0,"5058,5059,5060,5522,","5073,5074,5100,5530,5532,5546,5548,5549,5550,5...","5123,5170,5478,5566,5567,5568,5569,5570,5572,5...","5592,5593,5594,5766, 5588,5589,5590,5591",5.0,0.0,0.0,0.0,5.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,2.0,63.0,48.0,13.0,126.0,", 5668,5670","5064,5065,5066,5067,5077,5106,5155,5210,5227,5...","5180,5249,5250,5482,5484,5637,5640,5641,5642,5...","5254,5255,5256,5257,5258,5485,5487,5488,5869, ...",11.0,0.0,0.0,0.0,11.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,0.0,16.0,3.0,1.0,20.0,",","5044,5045,5050,5051,5053,5055,5056,5057,5631,5...","5903, 5129,5856",", 5126",1.0,0.0,0.0,0.0,1.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,2.0,43.0,27.0,15.0,87.0,"5369,5601,","5150,5152,5383,5400,5403,5404,5406,5445,5496,5...","5120,5182,5419,5420,5426,5428,5430,5477,5584,5...","5597,5655,5658, 5433,5435,5436,5660,5661,5758,...",6.0,0.0,0.0,0.0,6.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,0.0,23.0,21.0,3.0,47.0,",","5009,5010,5020,5021,5022,5341,5342,5344,5349,5...","5023,5024,5030,5031,5032,5033,5101,5356,5790,5...","5909, 5908,5910",3.0,0.0,0.0,0.0,3.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,0.0,30.0,5.0,8.0,43.0,",","5085,5086,5089,5090,5135,5136,5563,5696,5697,5...","5579,5580, 5117,5159,5581",", 5134,5583,5656,5767,6023,6024,6025,6027",3.0,0.0,0.0,0.0,3.0


# Reverse Type Logic 

In [61]:
diff_column_check=['amdifference','middaydifference','pmdifference','eveningdifference','totaldifference']
diff_column=check_all_characters_present(route_level_df,diff_column_check)
diff_column.sort()
diff_column

['AM_DIFFERENCE',
 'Evening_DIFFERENCE',
 'Midday_DIFFERENCE',
 'PM_DIFFERENCE',
 'Total_DIFFERENCE']

In [62]:
trip_oppo_dir_column_check=['tripinoppodir']
trip_oppo_dir_column=check_all_characters_present(df,trip_oppo_dir_column_check)
trip_oppo_dir_column

['TRIP_IN_OPPO_DIR']

In [63]:
route_survey_name_column_check=['routesurveyed']
route_survey_name_column=check_all_characters_present(df,route_survey_name_column_check)
route_survey_name_column

['ROUTE_SURVEYED']

In [64]:
oppo_dir_time_column_check=['oppodirtriptimecode']
oppo_dir_time_column=check_all_characters_present(df,oppo_dir_time_column_check)
oppo_dir_time_column

['OPPO_DIR_TRIP_TIMECode']

In [65]:
trip_code_column_check=['prevtransferscode','nexttransferscode']
trip_code_column=check_all_characters_present(df,trip_code_column_check)
trip_code_column.sort()
trip_code_column

['NEXT_TRANSFERSCode', 'PREV_TRANSFERSCode']

In [66]:
prev_trip_route_code_column_check=['tripfirstroutecode','tripsecondroutecode','tripthirdroutecode','tripfourthroutecode']
next_trip_route_code_column_check=['tripnextroutecode','tripafterroutecode','trip3rdroutecode','triplast4thrtecode']
prev_trip_route_code_column=check_all_characters_present(df,prev_trip_route_code_column_check)
next_trip_route_code_column=check_all_characters_present(df,next_trip_route_code_column_check)
# prev_trip_route_code_column.sort(),next_trip_route_code_column.sort()
prev_trip_route_code_column,next_trip_route_code_column

(['TRIP_FIRST_ROUTECode',
  'TRIP_SECOND_ROUTECode',
  'TRIP_THIRD_ROUTECode',
  'TRIP_FOURTH_ROUTECode'],
 ['TRIP_NEXT_ROUTECode',
  'TRIP_AFTER_ROUTECode',
  'TRIP_3RD_ROUTECode',
  'TRIP_LAST4TH_RTECode'])

In [67]:
df[[*prev_trip_route_code_column, *next_trip_route_code_column]]

,TRIP_FIRST_ROUTECode,TRIP_SECOND_ROUTECode,TRIP_THIRD_ROUTECode,TRIP_FOURTH_ROUTECode,TRIP_NEXT_ROUTECode,TRIP_AFTER_ROUTECode,TRIP_3RD_ROUTECode,TRIP_LAST4TH_RTECode
0,SAN_1_7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SAN_1_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SAN_1_12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
696,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
697,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
699,SAN_1_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [68]:
values_to_replace = ['-oth-']
df[[*prev_trip_route_code_column, *next_trip_route_code_column]] = df[
    [*prev_trip_route_code_column, *next_trip_route_code_column]
].replace(values_to_replace, np.nan)

In [69]:
df[[*prev_trip_route_code_column, *next_trip_route_code_column]]

,TRIP_FIRST_ROUTECode,TRIP_SECOND_ROUTECode,TRIP_THIRD_ROUTECode,TRIP_FOURTH_ROUTECode,TRIP_NEXT_ROUTECode,TRIP_AFTER_ROUTECode,TRIP_3RD_ROUTECode,TRIP_LAST4TH_RTECode
0,SAN_1_7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SAN_1_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SAN_1_12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
696,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
697,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
699,SAN_1_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [70]:
df[df[trip_oppo_dir_column[0]].str.lower()=='yes'][['id',*time_column,*route_survey_column,*route_survey_name_column,*oppo_dir_time_column]]

,id,TIME_ONCode,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,OPPO_DIR_TRIP_TIMECode
1,5005,MID1,SAN_1_14_00,14 - Inbound - To Bouquet Canyon/Plum Canyon,PM3
2,5006,MID1,SAN_1_14_00,14 - Inbound - To Bouquet Canyon/Plum Canyon,PM3
4,5010,MID2,SAN_1_14_01,14 - Outbound - To Newhall,PM2
7,5014,MID7,SAN_1_14_00,14 - Inbound - To Bouquet Canyon/Plum Canyon,MID6
10,5017,MID4,SAN_1_14_00,14 - Inbound - To Bouquet Canyon/Plum Canyon,MID4
...,...,...,...,...,...
682,6004,PM1,SAN_1_2_01,2 - Outbound - To Val Verde,PM7
683,6005,PM1,SAN_1_5_00,5 - Inbound - To Stevenson Ranch,MID2
684,6006,PM1,SAN_1_2_01,2 - Outbound - To Val Verde,PM7
685,6008,PM1,SAN_1_2_01,2 - Outbound - To Val Verde,PM7


In [71]:
reverse_df=df[df[trip_oppo_dir_column[0]].str.lower()=='yes'][['id',*route_survey_column,*route_survey_name_column,*trip_code_column,*prev_trip_route_code_column,*next_trip_route_code_column]]
# reverse_df=df[df[trip_oppo_dir_column[0]].str.lower()=='yes'][['id',*time_column,*route_survey_column,*route_survey_name_column,*oppo_dir_time_column]]

In [72]:
route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,0.0,10.0,15.0,7.0,32.0,",","5105,5157,5273,5275,5276,5277,5290,5291, 5278,...","5127,5292,5301,5302,5311,5312,5314,5795, 5307,...","5321,5323,5324,5325,5326, 5327,5328",3.0,0.0,0.0,0.0,3.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,0.0,18.0,6.0,2.0,26.0,",","5107,5148,5611,5679,5800,5835,5916,5935,5936, ...","5158,6003,6004,6006,6008,6009,","5967, 5966",3.0,0.0,0.0,0.0,3.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,0.0,13.0,5.0,0.0,18.0,",","5819,5993,5994,5995, 5043,5052,5054,5151,5439,...","5848,5849,5850, 5128,5901",",",2.0,0.0,0.0,2.0,4.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,0.0,30.0,26.0,0.0,56.0,",","5109,5110,5626,5634,5970,5972,5973,5985,5988, ...","5121,5125,5515,5517,5518,5651,5652,5653,5742,5...",",",3.0,0.0,0.0,3.0,6.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,4.0,61.0,26.0,8.0,99.0,"5058,5059,5060,5522,","5073,5074,5100,5530,5532,5546,5548,5549,5550,5...","5123,5170,5478,5566,5567,5568,5569,5570,5572,5...","5592,5593,5594,5766, 5588,5589,5590,5591",5.0,0.0,0.0,0.0,5.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,2.0,63.0,48.0,13.0,126.0,", 5668,5670","5064,5065,5066,5067,5077,5106,5155,5210,5227,5...","5180,5249,5250,5482,5484,5637,5640,5641,5642,5...","5254,5255,5256,5257,5258,5485,5487,5488,5869, ...",11.0,0.0,0.0,0.0,11.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,0.0,16.0,3.0,1.0,20.0,",","5044,5045,5050,5051,5053,5055,5056,5057,5631,5...","5903, 5129,5856",", 5126",1.0,0.0,0.0,0.0,1.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,2.0,43.0,27.0,15.0,87.0,"5369,5601,","5150,5152,5383,5400,5403,5404,5406,5445,5496,5...","5120,5182,5419,5420,5426,5428,5430,5477,5584,5...","5597,5655,5658, 5433,5435,5436,5660,5661,5758,...",6.0,0.0,0.0,0.0,6.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,0.0,23.0,21.0,3.0,47.0,",","5009,5010,5020,5021,5022,5341,5342,5344,5349,5...","5023,5024,5030,5031,5032,5033,5101,5356,5790,5...","5909, 5908,5910",3.0,0.0,0.0,0.0,3.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,0.0,30.0,5.0,8.0,43.0,",","5085,5086,5089,5090,5135,5136,5563,5696,5697,5...","5579,5580, 5117,5159,5581",", 5134,5583,5656,5767,6023,6024,6025,6027",3.0,0.0,0.0,0.0,3.0


In [73]:
route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,0.0,10.0,15.0,7.0,32.0,",","5105,5157,5273,5275,5276,5277,5290,5291, 5278,...","5127,5292,5301,5302,5311,5312,5314,5795, 5307,...","5321,5323,5324,5325,5326, 5327,5328",3.0,0.0,0.0,0.0,3.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,0.0,18.0,6.0,2.0,26.0,",","5107,5148,5611,5679,5800,5835,5916,5935,5936, ...","5158,6003,6004,6006,6008,6009,","5967, 5966",3.0,0.0,0.0,0.0,3.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,0.0,13.0,5.0,0.0,18.0,",","5819,5993,5994,5995, 5043,5052,5054,5151,5439,...","5848,5849,5850, 5128,5901",",",2.0,0.0,0.0,2.0,4.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,0.0,30.0,26.0,0.0,56.0,",","5109,5110,5626,5634,5970,5972,5973,5985,5988, ...","5121,5125,5515,5517,5518,5651,5652,5653,5742,5...",",",3.0,0.0,0.0,3.0,6.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,4.0,61.0,26.0,8.0,99.0,"5058,5059,5060,5522,","5073,5074,5100,5530,5532,5546,5548,5549,5550,5...","5123,5170,5478,5566,5567,5568,5569,5570,5572,5...","5592,5593,5594,5766, 5588,5589,5590,5591",5.0,0.0,0.0,0.0,5.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,2.0,63.0,48.0,13.0,126.0,", 5668,5670","5064,5065,5066,5067,5077,5106,5155,5210,5227,5...","5180,5249,5250,5482,5484,5637,5640,5641,5642,5...","5254,5255,5256,5257,5258,5485,5487,5488,5869, ...",11.0,0.0,0.0,0.0,11.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,0.0,16.0,3.0,1.0,20.0,",","5044,5045,5050,5051,5053,5055,5056,5057,5631,5...","5903, 5129,5856",", 5126",1.0,0.0,0.0,0.0,1.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,2.0,43.0,27.0,15.0,87.0,"5369,5601,","5150,5152,5383,5400,5403,5404,5406,5445,5496,5...","5120,5182,5419,5420,5426,5428,5430,5477,5584,5...","5597,5655,5658, 5433,5435,5436,5660,5661,5758,...",6.0,0.0,0.0,0.0,6.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,0.0,23.0,21.0,3.0,47.0,",","5009,5010,5020,5021,5022,5341,5342,5344,5349,5...","5023,5024,5030,5031,5032,5033,5101,5356,5790,5...","5909, 5908,5910",3.0,0.0,0.0,0.0,3.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,0.0,30.0,5.0,8.0,43.0,",","5085,5086,5089,5090,5135,5136,5563,5696,5697,5...","5579,5580, 5117,5159,5581",", 5134,5583,5656,5767,6023,6024,6025,6027",3.0,0.0,0.0,0.0,3.0


In [74]:
reverse_df.head()

,id,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,NEXT_TRANSFERSCode,PREV_TRANSFERSCode,TRIP_FIRST_ROUTECode,TRIP_SECOND_ROUTECode,TRIP_THIRD_ROUTECode,TRIP_FOURTH_ROUTECode,TRIP_NEXT_ROUTECode,TRIP_AFTER_ROUTECode,TRIP_3RD_ROUTECode,TRIP_LAST4TH_RTECode
1,5005,SAN_1_14_00,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5006,SAN_1_14_00,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5010,SAN_1_14_01,14 - Outbound - To Newhall,0.0,1.0,SAN_1_12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5014,SAN_1_14_00,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,5017,SAN_1_14_00,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
reverse_df[route_survey_column[0]]=reverse_df[route_survey_column[0]].apply(lambda x: '_'.join(x.split("_")[:-1]))

In [76]:
reverse_df.reset_index(inplace=True,drop=True)
reverse_df

,id,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,NEXT_TRANSFERSCode,PREV_TRANSFERSCode,TRIP_FIRST_ROUTECode,TRIP_SECOND_ROUTECode,TRIP_THIRD_ROUTECode,TRIP_FOURTH_ROUTECode,TRIP_NEXT_ROUTECode,TRIP_AFTER_ROUTECode,TRIP_3RD_ROUTECode,TRIP_LAST4TH_RTECode
0,5005,SAN_1_14,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5006,SAN_1_14,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5010,SAN_1_14,14 - Outbound - To Newhall,0.0,1.0,SAN_1_12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5014,SAN_1_14,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5017,SAN_1_14,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,6004,SAN_1_2,2 - Outbound - To Val Verde,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
413,6005,SAN_1_5,5 - Inbound - To Stevenson Ranch,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
414,6006,SAN_1_2,2 - Outbound - To Val Verde,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
415,6008,SAN_1_2,2 - Outbound - To Val Verde,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [77]:
df[(df[trip_code_column[1]]!=0)][[trip_code_column[1]]]

,PREV_TRANSFERSCode
0,1.0
3,1.0
4,1.0
9,1.0
19,1.0
...,...
623,1.0
628,1.0
653,1.0
669,1.0


In [78]:
# zero=0
# route_level_df.loc[route_level_df[route_survey_column[0]] == 'REN_1_VRGN','Total_DIFFERENCE']=zero
# route_level_df.loc[route_level_df[route_survey_column[0]] == 'REN_1_VRGN','Total_DIFFERENCE'].values
# route_level_df.head()

In [79]:
reverse_df[[*prev_trip_route_code_column,*next_trip_route_code_column]].fillna('',inplace=True)

In [80]:
# for index, row in reverse_df.iterrows():
#     random_value=random.choice([0,1])
#     value=int(route_level_df[(route_level_df[route_survey_column[0]]==row[route_survey_column[0]])]['Total_DIFFERENCE'].values)
#     prev_trans_value=int(df[(df['id']==row['id'])][trip_code_column[1]].values)
#     next_trans_value=int(df[(df['id']==row['id'])][trip_code_column[0]].values)
#     if random_value:
#         if value >0:
#             reverse_df.loc[index,'Type']='Reverse'
#             route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
#         elif prev_trans_value:
#             result_array = reverse_df[reverse_df['id'] == row['id']][prev_trip_route_code_column].values
#             values_in_list = result_array[0, :]
#             valid_values = [value for value in values_in_list if not (pd.isna(value) or value == '')]
#             prev_counter=0
#             for route in valid_values:
#                 prev_counter+=1
#                 value=int(route_level_df[(route_level_df[route_survey_column[0]]==route)]['Total_DIFFERENCE'].values)
#                 if value >0:
#                     reverse_df.loc[index,'Type']=f'Rev-p{prev_counter}'
#                     route_level_df.loc[route_level_df[route_survey_column[0]] == route,'Total_DIFFERENCE'] = value - 1
#                     break
#         elif next_trans_value:
#             result_array = reverse_df[reverse_df['id'] == row['id']][next_trip_route_code_column].values
#             values_in_list = result_array[0, :]
#             valid_values = [value for value in values_in_list if not (pd.isna(value) or value == '')]
#             next_counter=0
#             for route in valid_values:
#                 next_counter+=1
#                 value=int(route_level_df[(route_level_df[route_survey_column[0]]==route)]['Total_DIFFERENCE'].values)
#                 if value >0:
#                     reverse_df.loc[index,'Type']=f'Rev-n{next_counter}'
#                     route_level_df.loc[route_level_df[route_survey_column[0]] == route,'Total_DIFFERENCE'] = value - 1
#                     break
#         else:
#             reverse_df.loc[index,'Type']='Reverse'
#     else:
#         if prev_trans_value:
#             result_array = reverse_df[reverse_df['id'] == row['id']][prev_trip_route_code_column].values
#             values_in_list = result_array[0, :]
#             valid_values = [value for value in values_in_list if not (pd.isna(value) or value == '')]
#             prev_counter=0
#             for route in valid_values:
#                 prev_counter+=1
#                 value=int(route_level_df[(route_level_df[route_survey_column[0]]==route)]['Total_DIFFERENCE'].values)
#                 if value >0:
#                     reverse_df.loc[index,'Type']=f'Rev-p{prev_counter}'
#                     route_level_df.loc[route_level_df[route_survey_column[0]] == route,'Total_DIFFERENCE'] = value - 1
#                     break
#         elif next_trans_value:
#             result_array = reverse_df[reverse_df['id'] == row['id']][next_trip_route_code_column].values
#             values_in_list = result_array[0, :]
#             valid_values = [value for value in values_in_list if not (pd.isna(value) or value == '')]
#             next_counter=0
#             for route in valid_values:
#                 next_counter+=1
#                 value=int(route_level_df[(route_level_df[route_survey_column[0]]==route)]['Total_DIFFERENCE'].values)
#                 if value >0:
#                     reverse_df.loc[index,'Type']=f'Rev-n{next_counter}'
#                     route_level_df.loc[route_level_df[route_survey_column[0]] == route,'Total_DIFFERENCE'] = value - 1
#                     break
#         else:
#             reverse_df.loc[index,'Type']='Reverse'

In [81]:

# def get_valid_routes(row, route_code_column):
#     result_array = reverse_df[reverse_df['id'] == row['id']][route_code_column].values
#     values_in_list = result_array[0, :]
#     return [value for value in values_in_list if not (pd.isna(value) or value == '')]

# def process_route(route, counter_list, counter_prefix):
#     counter_list[0] += 1
#     rev_prefix=f'Rev-{counter_prefix}'
#     random_choice = random.choice([counter_prefix,rev_prefix ])
#     value = int(route_level_df[route_level_df[route_survey_column[0]] == route]['Total_DIFFERENCE'].values)  
#     if value > 0:
#         reverse_df.loc[index, 'Type'] = f'{random_choice}{counter_list[0]}'
#         route_level_df.loc[route_level_df[route_survey_column[0]] == route, 'Total_DIFFERENCE'] = value - 1
#         return True
#     return False


# for index, row in reverse_df.iterrows():
#     random_value = random.choice([0, 1])
#     value = int(route_level_df[route_level_df[route_survey_column[0]] == row[route_survey_column[0]]]['Total_DIFFERENCE'].values)
#     prev_trans_value = int(df[df['id'] == row['id']][trip_code_column[1]].values)
#     next_trans_value = int(df[df['id'] == row['id']][trip_code_column[0]].values)
#     # exit()
#     counter = [0]  # Use a list to store the counter value

#     if random_value:
#         if value > 0:
#             reverse_df.loc[index, 'Type'] = 'Reverse'
#             route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
#         elif prev_trans_value:
#             for route in get_valid_routes(row, prev_trip_route_code_column):
#                 result_value=process_route(route, counter, 'p')
#                 if result_value:
#                     break
#                 else:
#                     reverse_df.loc[index, 'Type'] = f'{random.choice(["p1","Rev-p1"])}'
#                     break
#         elif next_trans_value:
#             for route in get_valid_routes(row, next_trip_route_code_column):
#                 result_value=process_route(route, counter, 'n')
#                 if result_value:
#                     break
#                 else:
#                     reverse_df.loc[index, 'Type'] = f'{random.choice(["n1","Rev-n1"])}'
#                     break
#         else:
#             reverse_df.loc[index, 'Type'] = 'Reverse'
#     else:
#         if prev_trans_value:
#             for route in get_valid_routes(row, prev_trip_route_code_column):
#                 result_value=process_route(route, counter, 'p')
#                 if result_value:
#                     break
#                 else:
#                     reverse_df.loc[index, 'Type'] = f'{random.choice(["p1","Rev-p1"])}'
#                     break
#         elif next_trans_value:
#             for route in get_valid_routes(row, next_trip_route_code_column):
#                 result_value=process_route(route, counter, 'n')
#                 if result_value:
#                     break
#                 else:
#                     reverse_df.loc[index, 'Type'] = f'{random.choice(["n1","Rev-n1"])}'
#                     break
#         else:
#             reverse_df.loc[index, 'Type'] = 'Reverse'


In [82]:
def get_valid_routes(row, route_code_column):
    result_array = reverse_df[reverse_df['id'] == row['id']][route_code_column].values
    values_in_list = result_array[0, :]
    return [value for value in values_in_list if not (pd.isna(value) or value == '')]

def process_route(route, counter_list, counter_prefix):
    counter_list[0] += 1
    values = route_level_df[route_level_df[route_survey_column[0]] == route]['Total_DIFFERENCE'].values  
    if len(values) > 0:
        value=int(values)
        reverse_df.loc[index, 'Type'] = f'Rev-{counter_prefix}{counter_list[0]}'
        route_level_df.loc[route_level_df[route_survey_column[0]] == route, 'Total_DIFFERENCE'] = value - 1
        return True
    return False


for index, row in reverse_df.iterrows():
    random_value = random.choice([0, 1])
    value = int(route_level_df[route_level_df[route_survey_column[0]] == row[route_survey_column[0]]]['Total_DIFFERENCE'].values)
    prev_trans_value = int(df[df['id'] == row['id']][trip_code_column[1]].values)
    next_trans_value = int(df[df['id'] == row['id']][trip_code_column[0]].values)

    counter = [0]  # Use a list to store the counter value

    if value:
        if not random_value :
            reverse_df.loc[index, 'Type'] = 'Reverse'
            route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
        else:
            if prev_trans_value:
                for route in get_valid_routes(row, prev_trip_route_code_column):
                    result_value=process_route(route, counter, 'p')
                    if result_value:
                        break
            elif next_trans_value:
                for route in get_valid_routes(row, next_trip_route_code_column):
                    result_value=process_route(route, counter, 'n')
                    if result_value:
                        break
            else:
                reverse_df.loc[index, 'Type'] = 'Reverse'
    else:
        reverse_df.loc[index, 'Type'] = ''

In [83]:
route_level_df[route_level_df['ROUTE_SURVEYEDCode']=='REN_1_VRGN']

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE


In [84]:
# reverse_df['Type']='Reverse'
reverse_df['COMPLETED By']=''

In [85]:
reverse_df['Type'].fillna('Reverse',inplace=True)

In [86]:
reverse_df

,id,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,NEXT_TRANSFERSCode,PREV_TRANSFERSCode,TRIP_FIRST_ROUTECode,TRIP_SECOND_ROUTECode,TRIP_THIRD_ROUTECode,TRIP_FOURTH_ROUTECode,TRIP_NEXT_ROUTECode,TRIP_AFTER_ROUTECode,TRIP_3RD_ROUTECode,TRIP_LAST4TH_RTECode,Type,COMPLETED By
0,5005,SAN_1_14,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,
1,5006,SAN_1_14,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,
2,5010,SAN_1_14,14 - Outbound - To Newhall,0.0,1.0,SAN_1_12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,
3,5014,SAN_1_14,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,
4,5017,SAN_1_14,14 - Inbound - To Bouquet Canyon/Plum Canyon,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,6004,SAN_1_2,2 - Outbound - To Val Verde,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,
413,6005,SAN_1_5,5 - Inbound - To Stevenson Ranch,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,
414,6006,SAN_1_2,2 - Outbound - To Val Verde,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,
415,6008,SAN_1_2,2 - Outbound - To Val Verde,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,


In [87]:
reverse_df[reverse_df['Type']!=''].shape

(267, 15)

In [88]:
compare_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,0.0,10.0,15.0,7.0,32.0,",","5105,5157,5273,5275,5276,5277,5290,5291, 5278,...","5127,5292,5301,5302,5311,5312,5314,5795, 5307,...","5321,5323,5324,5325,5326, 5327,5328",3.0,0.0,0.0,0.0,-5.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,0.0,18.0,6.0,2.0,26.0,",","5107,5148,5611,5679,5800,5835,5916,5935,5936, ...","5158,6003,6004,6006,6008,6009,","5967, 5966",3.0,0.0,0.0,0.0,-4.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,0.0,13.0,5.0,0.0,18.0,",","5819,5993,5994,5995, 5043,5052,5054,5151,5439,...","5848,5849,5850, 5128,5901",",",2.0,0.0,0.0,2.0,0.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,0.0,30.0,26.0,0.0,56.0,",","5109,5110,5626,5634,5970,5972,5973,5985,5988, ...","5121,5125,5515,5517,5518,5651,5652,5653,5742,5...",",",3.0,0.0,0.0,3.0,0.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,4.0,61.0,26.0,8.0,99.0,"5058,5059,5060,5522,","5073,5074,5100,5530,5532,5546,5548,5549,5550,5...","5123,5170,5478,5566,5567,5568,5569,5570,5572,5...","5592,5593,5594,5766, 5588,5589,5590,5591",5.0,0.0,0.0,0.0,-28.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,2.0,63.0,48.0,13.0,126.0,", 5668,5670","5064,5065,5066,5067,5077,5106,5155,5210,5227,5...","5180,5249,5250,5482,5484,5637,5640,5641,5642,5...","5254,5255,5256,5257,5258,5485,5487,5488,5869, ...",11.0,0.0,0.0,0.0,-34.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,0.0,16.0,3.0,1.0,20.0,",","5044,5045,5050,5051,5053,5055,5056,5057,5631,5...","5903, 5129,5856",", 5126",1.0,0.0,0.0,0.0,0.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,2.0,43.0,27.0,15.0,87.0,"5369,5601,","5150,5152,5383,5400,5403,5404,5406,5445,5496,5...","5120,5182,5419,5420,5426,5428,5430,5477,5584,5...","5597,5655,5658, 5433,5435,5436,5660,5661,5758,...",6.0,0.0,0.0,0.0,-26.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,0.0,23.0,21.0,3.0,47.0,",","5009,5010,5020,5021,5022,5341,5342,5344,5349,5...","5023,5024,5030,5031,5032,5033,5101,5356,5790,5...","5909, 5908,5910",3.0,0.0,0.0,0.0,-12.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,0.0,30.0,5.0,8.0,43.0,",","5085,5086,5089,5090,5135,5136,5563,5696,5697,5...","5579,5580, 5117,5159,5581",", 5134,5583,5656,5767,6023,6024,6025,6027",3.0,0.0,0.0,0.0,0.0


In [89]:
reverse_df['Random']= np.random.randint(1000, 1000000, size=len(reverse_df))